In [13]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    
    if level < 2:
        print(root)
        for file in files:
            print("   ", file)

/kaggle/input
/kaggle/input/notebooks
/kaggle/input/datasets


In [14]:
import os

base_path = "/kaggle/input/datasets"

for root, dirs, files in os.walk(base_path):
    print("\nFolder:", root)
    for file in files:
        print("   File:", file)


Folder: /kaggle/input/datasets

Folder: /kaggle/input/datasets/gou14226

Folder: /kaggle/input/datasets/gou14226/m5-production-forecasting-data
   File: features_production_clean.parquet
   File: feature_scaler.pkl


In [15]:
import os
import pyarrow.parquet as pq

DATA_DIR = "/kaggle/input/datasets/gou14226/m5-production-forecasting-data"

FEATURE_FILE = os.path.join(
    DATA_DIR,
    "features_production_clean.parquet"
)

SCALER_FILE = os.path.join(
    DATA_DIR,
    "feature_scaler.pkl"
)

print("Feature file exists:", os.path.exists(FEATURE_FILE))
print("Scaler file exists:", os.path.exists(SCALER_FILE))

parquet_file = pq.ParquetFile(FEATURE_FILE)

print("\nParquet information:")
print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.metadata.num_columns)
print("Row groups:", parquet_file.num_row_groups)

print("\nFirst 10 columns:")
print(parquet_file.schema.names[:10])

Feature file exists: True
Scaler file exists: True

Parquet information:
Rows: 58327370
Columns: 39
Row groups: 61

First 10 columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk']


In [16]:
import pickle
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

# Load scaler
with open(SCALER_FILE, "rb") as f:
    scaler = pickle.load(f)

print("Scaler loaded successfully")
print("Number of features:", scaler.n_features_in_)

# Read only required columns from first row group
test_columns = [
    "item_id",
    "store_id",
    "date",
    "sales"
]

parquet_file = pq.ParquetFile(FEATURE_FILE)

test_df = parquet_file.read_row_group(
    0,
    columns=test_columns
).to_pandas()

print("\nTest row group:")
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nFirst 5 rows:")
print(test_df.head())

print("\nMissing values:")
print(test_df.isna().sum())

Scaler loaded successfully
Number of features: 19

Test row group:
Shape: (1048576, 4)
Columns: ['item_id', 'store_id', 'date', 'sales']

First 5 rows:
         item_id store_id       date  sales
0  HOBBIES_1_001     CA_1 2011-01-29      0
1  HOBBIES_1_002     CA_1 2011-01-29      0
2  HOBBIES_1_003     CA_1 2011-01-29      0
3  HOBBIES_1_004     CA_1 2011-01-29      0
4  HOBBIES_1_005     CA_1 2011-01-29      0

Missing values:
item_id     0
store_id    0
date        0
sales       0
dtype: int64


In [17]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def generate_batches_final(
    parquet_path,
    feature_columns,
    sequence_length=28,
    forecast_horizon=1,
    batch_size=256,
    selected_series=None,
    target_start_date=None,
    target_end_date=None
):
    """
    Memory-safe production sequence generator.
    """

    if target_start_date is not None:
        target_start_date = pd.Timestamp(target_start_date)

    if target_end_date is not None:
        target_end_date = pd.Timestamp(target_end_date)

    parquet_file = pq.ParquetFile(parquet_path)

    required_columns = [
        "item_id",
        "store_id",
        "date"
    ] + feature_columns

    selected_pairs = None

    if selected_series is not None:
        selected_pairs = set(selected_series)

    required_valid_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    history = {}

    X_batch = []
    y_batch = []

    for row_group in range(parquet_file.num_row_groups):

        df = parquet_file.read_row_group(
            row_group,
            columns=required_columns
        ).to_pandas()

        # Exact item-store filtering
        if selected_pairs is not None:

            df = df[
                df.apply(
                    lambda row: (
                        row["item_id"],
                        row["store_id"]
                    ) in selected_pairs,
                    axis=1
                )
            ]

        if len(df) == 0:
            continue

        df["date"] = pd.to_datetime(df["date"])

        # Process each series
        for key, current_series in df.groupby(
            ["item_id", "store_id"],
            sort=False
        ):

            current_series = (
                current_series
                .sort_values("date")
                .drop_duplicates("date")
                .reset_index(drop=True)
            )

            previous_length = 0

            # Add previous history
            if key in history:

                previous = history[key]

                previous_length = len(previous)

                combined = pd.concat(
                    [previous, current_series],
                    ignore_index=True
                )

            else:

                combined = current_series.copy()

            combined = (
                combined
                .drop_duplicates(subset=["date"])
                .sort_values("date")
                .reset_index(drop=True)
            )

            # Missing price handling
            if "sell_price" in combined.columns:

                combined["sell_price"] = (
                    combined["sell_price"]
                    .fillna(0)
                )

            # Generate new target dates
            if len(combined) >= (
                sequence_length + forecast_horizon
            ):

                max_target_index = (
                    len(combined)
                    - forecast_horizon
                )

                first_target_index = max(
                    sequence_length,
                    previous_length
                )

                for target_index in range(
                    first_target_index,
                    max_target_index + 1
                ):

                    target_date = combined.iloc[
                        target_index
                    ]["date"]

                    # Target date filtering
                    if (
                        target_start_date is not None
                        and target_date < target_start_date
                    ):
                        continue

                    if (
                        target_end_date is not None
                        and target_date > target_end_date
                    ):
                        continue

                    target_end = (
                        target_index
                        + forecast_horizon
                    )

                    X_window = combined.iloc[
                        target_index - sequence_length:
                        target_index
                    ][feature_columns]

                    y_window = combined.iloc[
                        target_index:
                        target_end
                    ]["sales"].values

                    # Skip incomplete lag/rolling inputs
                    if (
                        X_window[
                            required_valid_features
                        ]
                        .isna()
                        .any()
                        .any()
                    ):
                        continue

                    X_values = (
                        X_window
                        .values
                        .astype(np.float32)
                    )

                    y_values = (
                        y_window
                        .astype(np.float32)
                    )

                    # Numerical safety
                    if np.isnan(X_values).any():
                        continue

                    if np.isinf(X_values).any():
                        continue

                    if np.isnan(y_values).any():
                        continue

                    if np.isinf(y_values).any():
                        continue

                    X_batch.append(X_values)
                    y_batch.append(y_values)

                    # Yield full batch
                    if len(X_batch) == batch_size:

                        yield (
                            np.asarray(
                                X_batch,
                                dtype=np.float32
                            ),
                            np.asarray(
                                y_batch,
                                dtype=np.float32
                            )
                        )

                        X_batch = []
                        y_batch = []

            # Keep required history
            history[key] = combined.tail(
                sequence_length
            ).copy()

    # Final incomplete batch
    if X_batch:

        yield (
            np.asarray(
                X_batch,
                dtype=np.float32
            ),
            np.asarray(
                y_batch,
                dtype=np.float32
            )
        )

In [18]:
FEATURE_COLUMNS = [
    "sales",
    "sell_price",
    "price_available",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "is_event_day",
    "event_count",
    "snap_active"
]

VALIDATION_START_DATE = "2016-03-28"
VALIDATION_END_DATE = "2016-04-24"

generator = generate_batches_final(
    parquet_path=FEATURE_FILE,
    feature_columns=FEATURE_COLUMNS,
    sequence_length=28,
    forecast_horizon=1,
    batch_size=28,
    selected_series=[
        ("HOBBIES_1_001", "CA_1")
    ],
    target_start_date=VALIDATION_START_DATE,
    target_end_date=VALIDATION_END_DATE
)

X_val, y_val = next(generator)

print("X shape:", X_val.shape)
print("y shape:", y_val.shape)

print("Expected sequences:", 28)
print("Actual sequences:", len(X_val))

print("NaN in X:", np.isnan(X_val).sum())
print("Inf in X:", np.isinf(X_val).sum())

print("NaN in y:", np.isnan(y_val).sum())
print("Inf in y:", np.isinf(y_val).sum())

assert X_val.shape == (28, 28, 19)
assert y_val.shape == (28, 1)

print("\nVALIDATION GENERATOR TEST: PASS")

X shape: (28, 28, 19)
y shape: (28, 1)
Expected sequences: 28
Actual sequences: 28
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

VALIDATION GENERATOR TEST: PASS


In [19]:
import pickle
import numpy as np

# Load production scaler
with open(SCALER_FILE, "rb") as f:
    scaler = pickle.load(f)

# Scale validation sequences
X_2d = X_val.reshape(-1, len(FEATURE_COLUMNS))

X_scaled = scaler.transform(X_2d)

X_scaled = X_scaled.reshape(
    X_val.shape[0],
    X_val.shape[1],
    len(FEATURE_COLUMNS)
).astype(np.float32)

print("Original X shape:", X_val.shape)
print("Scaled X shape:", X_scaled.shape)
print("Scaled dtype:", X_scaled.dtype)

print("NaN in scaled X:", np.isnan(X_scaled).sum())
print("Inf in scaled X:", np.isinf(X_scaled).sum())

print("Scaled mean:", X_scaled.mean())
print("Scaled std:", X_scaled.std())

assert X_scaled.shape == (28, 28, 19)
assert X_scaled.dtype == np.float32
assert np.isnan(X_scaled).sum() == 0
assert np.isinf(X_scaled).sum() == 0

print("\nSCALER + GENERATOR TEST: PASS")

Original X shape: (28, 28, 19)
Scaled X shape: (28, 28, 19)
Scaled dtype: float32
NaN in scaled X: 0
Inf in scaled X: 0
Scaled mean: -0.045507032
Scaled std: 0.7409588

SCALER + GENERATOR TEST: PASS


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [10]:
import tensorflow as tf

SEQUENCE_LENGTH = 28
NUM_FEATURES = 19

def build_lstm_attention_model(
    sequence_length,
    num_features,
    lstm_units=64,
    dense_units=32,
    learning_rate=0.001
):
    inputs = tf.keras.Input(
        shape=(sequence_length, num_features),
        name="input_sequence"
    )

    lstm_output = tf.keras.layers.LSTM(
        lstm_units,
        return_sequences=True,
        name="lstm"
    )(inputs)

    attention_output = tf.keras.layers.Attention(
        name="attention"
    )([lstm_output, lstm_output])

    pooled_output = tf.keras.layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(attention_output)

    dense_output = tf.keras.layers.Dense(
        dense_units,
        activation="relu",
        name="dense"
    )(pooled_output)

    output = tf.keras.layers.Dense(
        1,
        name="sales_forecast"
    )(dense_output)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=output,
        name="lstm_attention_forecaster"
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model


model = build_lstm_attention_model(
    sequence_length=SEQUENCE_LENGTH,
    num_features=NUM_FEATURES
)

model.summary()

Model: "lstm_attention_forecaster"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 28, 19)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 28, 64)    │     21,504 │ input_sequence[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 28, 64)    │          0 │ lstm[0][0],       │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sales_forecast      │ (None, 1)         │         33 │ dense[0][0]       │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,617 (92.25 KB)

 Trainable params: 23,617 (92.25 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
predictions = model.predict(
    X_scaled,
    verbose=0
)

print("Prediction shape:", predictions.shape)
print("First 5 predictions:")
print(predictions[:5].flatten())

print("NaN in predictions:", np.isnan(predictions).sum())
print("Inf in predictions:", np.isinf(predictions).sum())

assert predictions.shape == (28, 1)
assert np.isnan(predictions).sum() == 0
assert np.isinf(predictions).sum() == 0

print("\nMODEL INFERENCE TEST: PASS")

Prediction shape: (28, 1)
First 5 predictions:
[-0.03803625 -0.0358278  -0.03235578 -0.0294207  -0.02803938]
NaN in predictions: 0
Inf in predictions: 0

MODEL INFERENCE TEST: PASS


In [20]:
def scaled_batch_generator(
    parquet_path,
    feature_columns,
    scaler,
    sequence_length,
    forecast_horizon,
    batch_size,
    selected_series=None,
    target_start_date=None,
    target_end_date=None
):
    
    base_generator = generate_batches_final(
        parquet_path=parquet_path,
        feature_columns=feature_columns,
        sequence_length=sequence_length,
        forecast_horizon=forecast_horizon,
        batch_size=batch_size,
        selected_series=selected_series,
        target_start_date=target_start_date,
        target_end_date=target_end_date
    )

    for X_batch, y_batch in base_generator:

        actual_batch_size = X_batch.shape[0]

        X_2d = X_batch.reshape(
            -1,
            len(feature_columns)
        )

        X_df = pd.DataFrame(
            X_2d,
            columns=feature_columns
        )

        X_scaled = scaler.transform(X_df)

        X_scaled = X_scaled.reshape(
            actual_batch_size,
            sequence_length,
            len(feature_columns)
        ).astype(np.float32)

        yield (
            X_scaled,
            y_batch.astype(np.float32)
        )


train_generator = scaled_batch_generator(
    parquet_path=FEATURE_FILE,
    feature_columns=FEATURE_COLUMNS,
    scaler=scaler,
    sequence_length=28,
    forecast_horizon=1,
    batch_size=64,
    target_end_date="2016-03-27"
)

X_train_test, y_train_test = next(train_generator)

print("X shape:", X_train_test.shape)
print("y shape:", y_train_test.shape)

print("dtype:", X_train_test.dtype)

print("NaN in X:", np.isnan(X_train_test).sum())
print("Inf in X:", np.isinf(X_train_test).sum())

print("NaN in y:", np.isnan(y_train_test).sum())
print("Inf in y:", np.isinf(y_train_test).sum())

assert X_train_test.shape == (64, 28, 19)
assert y_train_test.shape == (64, 1)

assert np.isnan(X_train_test).sum() == 0
assert np.isinf(X_train_test).sum() == 0

print("\nTRAINING GENERATOR TEST: PASS")

X shape: (64, 28, 19)
y shape: (64, 1)
dtype: float32
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

TRAINING GENERATOR TEST: PASS


In [21]:
OUTPUT_SIGNATURE = (
    tf.TensorSpec(
        shape=(None, 28, 19),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(None, 1),
        dtype=tf.float32
    )
)


def train_data_generator():
    for X_batch, y_batch in scaled_batch_generator(
        parquet_path=FEATURE_FILE,
        feature_columns=FEATURE_COLUMNS,
        scaler=scaler,
        sequence_length=28,
        forecast_horizon=1,
        batch_size=64,
        target_end_date="2016-03-27"
    ):
        yield X_batch, y_batch


train_dataset = tf.data.Dataset.from_generator(
    train_data_generator,
    output_signature=OUTPUT_SIGNATURE
)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)


train_batch = next(iter(train_dataset))

X_batch, y_batch = train_batch

print("X shape:", X_batch.shape)
print("y shape:", y_batch.shape)
print("X dtype:", X_batch.dtype)
print("y dtype:", y_batch.dtype)

print("NaN in X:", tf.reduce_sum(tf.cast(tf.math.is_nan(X_batch), tf.int32)).numpy())
print("Inf in X:", tf.reduce_sum(tf.cast(tf.math.is_inf(X_batch), tf.int32)).numpy())

print("NaN in y:", tf.reduce_sum(tf.cast(tf.math.is_nan(y_batch), tf.int32)).numpy())
print("Inf in y:", tf.reduce_sum(tf.cast(tf.math.is_inf(y_batch), tf.int32)).numpy())

assert X_batch.shape == (64, 28, 19)
assert y_batch.shape == (64, 1)

print("\nTF.DATA TRAINING PIPELINE TEST: PASS")

X shape: (64, 28, 19)
y shape: (64, 1)
X dtype: <dtype: 'float32'>
y dtype: <dtype: 'float32'>
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

TF.DATA TRAINING PIPELINE TEST: PASS


In [22]:
def validation_data_generator():
    for X_batch, y_batch in scaled_batch_generator(
        parquet_path=FEATURE_FILE,
        feature_columns=FEATURE_COLUMNS,
        scaler=scaler,
        sequence_length=28,
        forecast_horizon=1,
        batch_size=28,
        target_start_date="2016-03-28",
        target_end_date="2016-04-24"
    ):
        yield X_batch, y_batch


validation_dataset = tf.data.Dataset.from_generator(
    validation_data_generator,
    output_signature=(
        tf.TensorSpec(
            shape=(None, 28, 19),
            dtype=tf.float32
        ),
        tf.TensorSpec(
            shape=(None, 1),
            dtype=tf.float32
        )
    )
)

validation_dataset = validation_dataset.prefetch(
    tf.data.AUTOTUNE
)


X_val_batch, y_val_batch = next(
    iter(validation_dataset)
)

print("X shape:", X_val_batch.shape)
print("y shape:", y_val_batch.shape)

print("X dtype:", X_val_batch.dtype)
print("y dtype:", y_val_batch.dtype)

print(
    "NaN in X:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(X_val_batch), tf.int32)
    ).numpy()
)

print(
    "Inf in X:",
    tf.reduce_sum(
        tf.cast(tf.math.is_inf(X_val_batch), tf.int32)
    ).numpy()
)

print(
    "NaN in y:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(y_val_batch), tf.int32)
    ).numpy()
)

print(
    "Inf in y:",
    tf.reduce_sum(
        tf.cast(tf.math.is_inf(y_val_batch), tf.int32)
    ).numpy()
)

assert X_val_batch.shape == (28, 28, 19)
assert y_val_batch.shape == (28, 1)

print("\nVALIDATION TF.DATA PIPELINE TEST: PASS")

X shape: (28, 28, 19)
y shape: (28, 1)
X dtype: <dtype: 'float32'>
y dtype: <dtype: 'float32'>
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

VALIDATION TF.DATA PIPELINE TEST: PASS


In [23]:
import os

MODEL_DIR = "/kaggle/working/models"
REPORT_DIR = "/kaggle/working/reports"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "lstm_attention_production.keras"
)

HISTORY_PATH = os.path.join(
    REPORT_DIR,
    "training_history.csv"
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=MODEL_PATH,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

csv_logger = tf.keras.callbacks.CSVLogger(
    HISTORY_PATH,
    append=False
)

callbacks = [
    early_stopping,
    model_checkpoint,
    csv_logger
]

print("Model path:", MODEL_PATH)
print("History path:", HISTORY_PATH)
print("Callbacks:", len(callbacks))

print("\nTRAINING CALLBACK SETUP: PASS")

Model path: /kaggle/working/models/lstm_attention_production.keras
History path: /kaggle/working/reports/training_history.csv
Callbacks: 3

TRAINING CALLBACK SETUP: PASS


In [24]:
print("=" * 60)
print("FINAL TRAINING CONFIGURATION")
print("=" * 60)

print("Feature file:")
print(FEATURE_FILE)

print("\nScaler file:")
print(SCALER_FILE)

print("\nModel:")
print(model.name)

print("\nSequence length:", SEQUENCE_LENGTH)
print("Number of features:", NUM_FEATURES)
print("Forecast horizon:", 1)

print("\nTraining end date:", "2016-03-27")
print("Validation start date:", "2016-03-28")
print("Validation end date:", "2016-04-24")

print("\nBatch size:", 64)
print("Epochs:", 10)
print("Training steps per epoch:", 5000)
print("Validation steps:", 500)

print("\nLoss:", "MSE")
print("Metric:", "MAE")
print("Optimizer:", "Adam")
print("Learning rate:", 0.001)

print("\nModel parameters:", model.count_params())

print("\nCallbacks:")
for callback in callbacks:
    print(" -", callback.__class__.__name__)

print("\n" + "=" * 60)
print("FINAL CONFIGURATION CHECK: PASS")
print("=" * 60)

FINAL TRAINING CONFIGURATION
Feature file:
/kaggle/input/datasets/gou14226/m5-production-forecasting-data/features_production_clean.parquet

Scaler file:
/kaggle/input/datasets/gou14226/m5-production-forecasting-data/feature_scaler.pkl

Model:
lstm_attention_forecaster

Sequence length: 28
Number of features: 19
Forecast horizon: 1

Training end date: 2016-03-27
Validation start date: 2016-03-28
Validation end date: 2016-04-24

Batch size: 64
Epochs: 10
Training steps per epoch: 5000
Validation steps: 500

Loss: MSE
Metric: MAE
Optimizer: Adam
Learning rate: 0.001

Model parameters: 23617

Callbacks:
 - EarlyStopping
 - ModelCheckpoint
 - CSVLogger

FINAL CONFIGURATION CHECK: PASS


In [18]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10,
    steps_per_epoch=5000,
    validation_steps=500,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - loss: 6.3116 - mae: 0.9114
Epoch 1: val_loss improved from None to 4.16213, saving model to /kaggle/working/models/lstm_attention_production.keras

Epoch 1: finished saving model to /kaggle/working/models/lstm_attention_production.keras
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 675s 135ms/step - loss: 6.2404 - mae: 0.8688 - val_loss: 4.1621 - val_mae: 1.1323
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - loss: 6.7597 - mae: 0.7540
Epoch 2: val_loss improved from 4.16213 to 3.83022, saving model to /kaggle/working/models/lstm_attention_production.keras

Epoch 2: finished saving model to /kaggle/working/models/lstm_attention_production.keras
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 676s 135ms/step - loss: 3.5277 - mae: 0.5754 - val_loss: 3.8302 - val_mae: 1.1179
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - loss: 2.0461 - mae: 0.7390
Epoch 3: val_loss did not improve from 3.83022
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 674s 135ms/step - 

In [19]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np

MODEL_PATH = "/kaggle/working/models/lstm_attention_production.keras"
HISTORY_PATH = "/kaggle/working/reports/training_history.csv"

print("=" * 60)
print("SAVED MODEL CHECK")
print("=" * 60)

print("\nModel exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)
    print("Model size (MB):", round(size_mb, 2))

print("\nHistory exists:", os.path.exists(HISTORY_PATH))

if os.path.exists(HISTORY_PATH):
    history_df = pd.read_csv(HISTORY_PATH)
    print("History shape:", history_df.shape)
    print("\nTraining history:")
    print(history_df)

print("\nLoading saved model...")

saved_model = tf.keras.models.load_model(MODEL_PATH)

print("Model name:", saved_model.name)
print("Parameters:", saved_model.count_params())

print("\nRunning prediction...")

predictions = saved_model.predict(
    X_val_batch,
    verbose=0
)

print("Prediction shape:", predictions.shape)
print("First 5 predictions:")
print(predictions[:5].flatten())

print("NaN:", np.isnan(predictions).sum())
print("Inf:", np.isinf(predictions).sum())

assert os.path.exists(MODEL_PATH)
assert predictions.shape == (28, 1)
assert np.isnan(predictions).sum() == 0
assert np.isinf(predictions).sum() == 0
assert saved_model.count_params() == 23617

print("\nSAVED MODEL CHECK: PASS")

SAVED MODEL CHECK

Model exists: True
Model size (MB): 0.31

History exists: True
History shape: (8, 5)

Training history:
   epoch      loss       mae  val_loss   val_mae
0      0  6.240382  0.868832  4.162128  1.132254
1      1  3.527693  0.575439  3.830218  1.117857
2      2  1.767140  0.670764  3.896049  1.143475
3      3  2.514004  0.937990  3.577513  1.051957
4      4  4.705852  0.987614  3.802109  1.100851
5      5  3.359369  0.726215  3.563063  1.084482
6      6  0.422035  0.368589  3.621961  1.041879
7      7  1.921497  0.621162  3.640879  1.040920

Loading saved model...
Model name: lstm_attention_forecaster
Parameters: 23617

Running prediction...
Prediction shape: (28, 1)
First 5 predictions:
[0.5563291  0.5559412  0.53865594 0.5193678  0.63905746]
NaN: 0
Inf: 0

SAVED MODEL CHECK: PASS


In [20]:
import numpy as np
import tensorflow as tf

print("=" * 60)
print("FULL VALIDATION EVALUATION")
print("=" * 60)

# Best saved production model
model = tf.keras.models.load_model(MODEL_PATH)

# Total validation sequences
TOTAL_VALIDATION_SEQUENCES = 30490 * 28

BATCH_SIZE_EVAL = 64
TOTAL_BATCHES = int(
    np.ceil(TOTAL_VALIDATION_SEQUENCES / BATCH_SIZE_EVAL)
)

print("\nValidation sequences:", TOTAL_VALIDATION_SEQUENCES)
print("Batch size:", BATCH_SIZE_EVAL)
print("Validation batches:", TOTAL_BATCHES)

# Accumulators
absolute_error_sum = 0.0
squared_error_sum = 0.0
actual_sum = 0.0
total_count = 0

print("\nStarting validation prediction...")

for batch_number, (X_batch, y_batch) in enumerate(
    validation_dataset
):

    # Stop after all validation sequences
    if batch_number >= TOTAL_BATCHES:
        break

    predictions = model.predict(
        X_batch,
        verbose=0
    )

    y_true = y_batch.numpy().reshape(-1)
    y_pred = predictions.reshape(-1)

    errors = y_true - y_pred

    absolute_error_sum += np.sum(
        np.abs(errors)
    )

    squared_error_sum += np.sum(
        errors ** 2
    )

    actual_sum += np.sum(
        np.abs(y_true)
    )

    total_count += len(y_true)

    if (batch_number + 1) % 1000 == 0:
        print(
            f"Processed batches: "
            f"{batch_number + 1}/{TOTAL_BATCHES}"
        )

# Metrics
mae = absolute_error_sum / total_count

rmse = np.sqrt(
    squared_error_sum / total_count
)

wape = (
    absolute_error_sum / actual_sum
) * 100

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

print("Total sequences evaluated:", total_count)
print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("WAPE:", round(wape, 4), "%")

print("\nFULL VALIDATION EVALUATION: COMPLETE")

FULL VALIDATION EVALUATION

Validation sequences: 853720
Batch size: 64
Validation batches: 13340

Starting validation prediction...
Processed batches: 1000/13340
Processed batches: 2000/13340
Processed batches: 3000/13340
Processed batches: 4000/13340
Processed batches: 5000/13340
Processed batches: 6000/13340
Processed batches: 7000/13340
Processed batches: 8000/13340
Processed batches: 9000/13340
Processed batches: 10000/13340
Processed batches: 11000/13340
Processed batches: 12000/13340
Processed batches: 13000/13340

VALIDATION RESULTS
Total sequences evaluated: 373520
MAE : 1.0828
RMSE: 2.3498
WAPE: 76.8203 %

FULL VALIDATION EVALUATION: COMPLETE


In [3]:
import glob
import os

model_files = glob.glob(
    "/kaggle/input/**/lstm_attention_production.keras",
    recursive=True
)

print("Found model files:")
for path in model_files:
    print(path)

assert len(model_files) > 0, "Model file not found!"

MODEL_PATH = model_files[0]

print("\nMODEL_PATH:")
print(MODEL_PATH)

print("\nModel exists:", os.path.exists(MODEL_PATH))
print("Model size (MB):", round(os.path.getsize(MODEL_PATH) / (1024 * 1024), 2))

Found model files:
/kaggle/input/notebooks/gou14226/lstm-attention-production/models/lstm_attention_production.keras

MODEL_PATH:
/kaggle/input/notebooks/gou14226/lstm-attention-production/models/lstm_attention_production.keras

Model exists: True
Model size (MB): 0.31


In [4]:
import tensorflow as tf

print("=" * 60)
print("LOADING SAVED PRODUCTION MODEL")
print("=" * 60)

model = tf.keras.models.load_model(MODEL_PATH)

print("\nModel loaded successfully")
print("Model name:", model.name)
print("Parameters:", model.count_params())

print("\nInput shape:", model.input_shape)
print("Output shape:", model.output_shape)

print("\nMODEL LOAD TEST: PASS")

LOADING SAVED PRODUCTION MODEL


I0000 00:00:1789905051.032826      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789905051.035642      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



Model loaded successfully
Model name: lstm_attention_forecaster
Parameters: 23617

Input shape: (None, 28, 19)
Output shape: (None, 1)

MODEL LOAD TEST: PASS


In [26]:
import os

MODEL_PATH = "/kaggle/input/notebooks/gou14226/lstm-attention-production/models/lstm_attention_production.keras"

print("MODEL_PATH:")
print(MODEL_PATH)

print("\nModel exists:", os.path.exists(MODEL_PATH))

assert os.path.exists(MODEL_PATH)

print("\nMODEL PATH CHECK: PASS")

MODEL_PATH:
/kaggle/input/notebooks/gou14226/lstm-attention-production/models/lstm_attention_production.keras

Model exists: True

MODEL PATH CHECK: PASS


In [27]:
import numpy as np
import tensorflow as tf

print("=" * 60)
print("FULL VALIDATION EVALUATION - CORRECTED")
print("=" * 60)

# Load best saved model
model = tf.keras.models.load_model(MODEL_PATH)

absolute_error_sum = 0.0
squared_error_sum = 0.0
actual_sum = 0.0
total_count = 0
batch_count = 0

print("\nStarting validation prediction...")

for X_batch, y_batch in validation_dataset:

    predictions = model.predict(
        X_batch,
        verbose=0
    )

    y_true = y_batch.numpy().reshape(-1)
    y_pred = predictions.reshape(-1)

    errors = y_true - y_pred

    absolute_error_sum += np.sum(
        np.abs(errors)
    )

    squared_error_sum += np.sum(
        errors ** 2
    )

    actual_sum += np.sum(
        np.abs(y_true)
    )

    total_count += len(y_true)
    batch_count += 1

    if batch_count % 5000 == 0:
        print(
            f"Processed batches: {batch_count} | "
            f"Sequences: {total_count}"
        )

# Calculate metrics
mae = absolute_error_sum / total_count

rmse = np.sqrt(
    squared_error_sum / total_count
)

wape = (
    absolute_error_sum / actual_sum
) * 100

print("\n" + "=" * 60)
print("FINAL VALIDATION RESULTS")
print("=" * 60)

print("Total batches evaluated:", batch_count)
print("Total sequences evaluated:", total_count)

print("\nMAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("WAPE:", round(wape, 4), "%")

print("\nExpected validation sequences:", 853720)
print("Actual validation sequences:", total_count)

assert total_count == 853720

print("\nFULL VALIDATION EVALUATION: PASS")

FULL VALIDATION EVALUATION - CORRECTED

Starting validation prediction...
Processed batches: 5000 | Sequences: 140000
Processed batches: 10000 | Sequences: 280000
Processed batches: 15000 | Sequences: 420000
Processed batches: 20000 | Sequences: 560000
Processed batches: 25000 | Sequences: 700000
Processed batches: 30000 | Sequences: 840000

FINAL VALIDATION RESULTS
Total batches evaluated: 30490
Total sequences evaluated: 853720

MAE : 1.0564
RMSE: 2.3925
WAPE: 76.1988 %

Expected validation sequences: 853720
Actual validation sequences: 853720

FULL VALIDATION EVALUATION: PASS
